In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/paraphrase-MiniLM-L3-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 256 if device == "mps" else 64
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})

In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["label"] = df["label"].astype(np.float32)

df["sentence1"] = df["sentence1"].astype(str)
df["sentence2"] = df["sentence2"].astype(str)
df["char_len_1"] = df["sentence1"].str.len().astype(np.int32)
df["char_len_2"] = df["sentence2"].str.len().astype(np.int32)
df["tok_len_1"] = df["sentence1"].str.split().str.len().astype(np.int32)
df["tok_len_2"] = df["sentence2"].str.split().str.len().astype(np.int32)
df["mean_tok_len"] = ((df["tok_len_1"] + df["tok_len_2"]) / 2.0).astype(np.float32)
df["max_tok_len"] = df[["tok_len_1", "tok_len_2"]].max(axis=1).astype(np.int32)
df["min_tok_len"] = df[["tok_len_1", "tok_len_2"]].min(axis=1).astype(np.int32)

medium_length_mask = (
    df["tok_len_1"].between(8, 18, inclusive="both")
    & df["tok_len_2"].between(8, 18, inclusive="both")
)
subset_df = df.loc[medium_length_mask].copy().reset_index(drop=True)

def length_stratum(x):
    if x <= 10:
        return "short-medium"
    if x <= 14:
        return "medium"
    return "long-medium"

subset_df["length_stratum"] = subset_df["mean_tok_len"].apply(length_stratum)
subset_df["token_length_gap"] = (subset_df["tok_len_1"] - subset_df["tok_len_2"]).abs().astype(np.int32)

print({
    "num_examples_full_validation": int(len(df)),
    "num_examples_medium_length_subset": int(len(subset_df)),
    "length_rule": "keep pairs where both sentence token lengths are between 8 and 18 inclusive",
    "length_strata": subset_df["length_stratum"].value_counts().sort_index().to_dict(),
})
print(subset_df[["sentence1", "sentence2", "label", "tok_len_1", "tok_len_2", "mean_tok_len", "length_stratum"]].head())

In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()
print(model_name)

In [ ]:
sentences1 = subset_df["sentence1"].tolist()
sentences2 = subset_df["sentence2"].tolist()
labels = subset_df["label"].to_numpy(dtype=np.float32)

emb1 = model.encode(
    sentences1,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

emb2 = model.encode(
    sentences2,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

cosine_similarity = np.sum(emb1 * emb2, axis=1).astype(np.float32)
predicted_score_0_5 = (2.5 * (cosine_similarity + 1.0)).astype(np.float32)
absolute_error = np.abs(predicted_score_0_5 - labels).astype(np.float32)
squared_error = np.square(predicted_score_0_5 - labels).astype(np.float32)

results_df = subset_df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["absolute_error"] = absolute_error
results_df["squared_error"] = squared_error
results_df["signed_error"] = (results_df["predicted_score_0_5"] - results_df["label"]).astype(np.float32)
results_df["agreement_gap"] = np.abs(results_df["cosine_similarity"] - (results_df["label"] / 2.5 - 1.0)).astype(np.float32)

print(results_df[["sentence1", "sentence2", "label", "tok_len_1", "tok_len_2", "length_stratum", "cosine_similarity", "predicted_score_0_5", "absolute_error"]].head(10))

In [ ]:
pearson_corr = pearsonr(results_df["predicted_score_0_5"], results_df["label"]).statistic
spearman_corr = spearmanr(results_df["predicted_score_0_5"], results_df["label"]).statistic
mae = float(results_df["absolute_error"].mean())
rmse = float(np.sqrt(results_df["squared_error"].mean()))

label_bin_edges = [-0.001, 1.0, 2.0, 3.0, 4.0, 5.001]
label_bin_names = ["[0,1)", "[1,2)", "[2,3)", "[3,4)", "[4,5]"]
results_df["label_bin"] = pd.cut(
    results_df["label"],
    bins=label_bin_edges,
    labels=label_bin_names,
    include_lowest=True,
    right=False,
)

length_stratum_agg = (
    results_df.groupby("length_stratum", observed=False)
    .agg(
        count=("label", "size"),
        label_mean=("label", "mean"),
        pred_mean=("predicted_score_0_5", "mean"),
        cosine_mean=("cosine_similarity", "mean"),
        mean_tok_len=("mean_tok_len", "mean"),
        mean_token_gap=("token_length_gap", "mean"),
        mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: float(np.sqrt(np.mean(x)))),
        signed_error_mean=("signed_error", "mean"),
    )
    .reset_index()
)

label_bin_agg = (
    results_df.groupby("label_bin", observed=False)
    .agg(
        count=("label", "size"),
        label_mean=("label", "mean"),
        pred_mean=("predicted_score_0_5", "mean"),
        mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: float(np.sqrt(np.mean(x)))),
        mean_tok_len=("mean_tok_len", "mean"),
    )
    .reset_index()
)

token_gap_edges = [-0.1, 0.5, 2.5, 100.0]
token_gap_names = ["same_length", "small_gap_1_2", "large_gap_3_plus"]
results_df["token_gap_bin"] = pd.cut(
    results_df["token_length_gap"],
    bins=token_gap_edges,
    labels=token_gap_names,
    include_lowest=True,
    right=True,
)

token_gap_agg = (
    results_df.groupby("token_gap_bin", observed=False)
    .agg(
        count=("label", "size"),
        mean_gap=("token_length_gap", "mean"),
        label_mean=("label", "mean"),
        pred_mean=("predicted_score_0_5", "mean"),
        mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: float(np.sqrt(np.mean(x)))),
        signed_error_mean=("signed_error", "mean"),
    )
    .reset_index()
)

token_length_corr = {
    "corr_abs_error_with_mean_tok_len": round(float(np.corrcoef(results_df["mean_tok_len"], results_df["absolute_error"])[0, 1]), 6),
    "corr_abs_error_with_token_length_gap": round(float(np.corrcoef(results_df["token_length_gap"], results_df["absolute_error"])[0, 1]), 6),
}

print({
    "pearson_correlation": round(float(pearson_corr), 6),
    "spearman_correlation": round(float(spearman_corr), 6),
    "mae_0_5": round(mae, 6),
    "rmse_0_5": round(rmse, 6),
    **token_length_corr,
})
print("PER_LENGTH_STRATUM")
print(length_stratum_agg)
print("\nPER_LABEL_BIN")
print(label_bin_agg)
print("\nTOKEN_LENGTH_ERROR_ANALYSIS")
print(token_gap_agg)

In [ ]:
best_examples = (
    results_df.sort_values(
        by=["absolute_error", "agreement_gap", "mean_tok_len", "label"],
        ascending=[True, True, True, False],
    )
    [["sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity", "tok_len_1", "tok_len_2", "token_length_gap", "length_stratum", "absolute_error", "signed_error"]]
    .head(10)
    .reset_index(drop=True)
)

worst_examples = (
    results_df.sort_values(
        by=["absolute_error", "agreement_gap", "mean_tok_len", "label"],
        ascending=[False, False, False, False],
    )
    [["sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity", "tok_len_1", "tok_len_2", "token_length_gap", "length_stratum", "absolute_error", "signed_error"]]
    .head(10)
    .reset_index(drop=True)
)

pd.set_option("display.max_colwidth", 160)
print("BEST_AGREEMENT_EXAMPLES")
print(best_examples)
print("\nWORST_AGREEMENT_EXAMPLES")
print(worst_examples)

In [ ]:
runtime_seconds = time.time() - start_time

summary = {
    "device_used": device,
    "model_name": model_name,
    "dataset_split": f"{dataset_name}/{dataset_config}/{split_name}",
    "subset_rule": "both sentences have token length between 8 and 18 inclusive",
    "num_examples_full_validation": int(len(df)),
    "num_examples_subset": int(len(results_df)),
    "pearson_correlation": round(float(pearson_corr), 6),
    "spearman_correlation": round(float(spearman_corr), 6),
    "mae_0_5": round(mae, 6),
    "rmse_0_5": round(rmse, 6),
    "runtime_seconds": round(float(runtime_seconds), 2),
}

print(summary)